In [1]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"

import jax
print("JAX device:", jax.devices())
jax.config.update('jax_disable_jit', False) # Turn off JIT because of an issue in shortwave_radiation.py:169
jax.config.update("jax_debug_infs", True) # doesn't add any time since the saved time is otherwise spent getting the nodal quantities
jax.config.update("jax_debug_nans", False) # some physics fields might be nan

JAX device: [CpuDevice(id=0)]


In [2]:
import sys
from pathlib import Path

paths_check = [
    (Path(os.path.abspath(".")) / ".." / ".." / "jax-gcm").resolve(),
    (Path(os.path.abspath(".")) / "..").resolve(),
]

for module_path in paths_check:
    module_path = str(module_path)
    if module_path in sys.path:
        print("Path exist: ", module_path)
    else:
        print("Add Path: ", module_path)
        sys.path.append(module_path)


Add Path:  /home/t2hsu/projects/jax-gcm
Add Path:  /home/t2hsu/projects/jax-esm


In [3]:
import numpy as np
import xarray as xr
import pandas as pd

#import jcm
#from jcm.model import Model, get_coords
#from jcm.boundaries import initialize_boundaries

from jax_esm.Master import Master
from jax_esm.components.base import ComponentConfig
from jax_esm.components.Speedy import Speedy
from jax_esm.components.SlabOceanModel import SlabOceanModel
from jax_esm.components.FluxModel import FluxModel

Path exist:  /home/t2hsu/projects/jax-gcm
Path exist:  /home/t2hsu/projects/jax-esm


## Create Model

In [4]:
total_simulation_time =  60 * 86400.0 
master_time_step      =  24 *  3600.0  # sec
master_steps = int(total_simulation_time / master_time_step)

atm_substeps          =  24           # count
atm_save_interval     =  master_time_step # sec

ocn_substeps          =  1            # count
ocn_save_interval     = 24 *  3600.0  # sec

flx_substeps          =  1            # count
flx_save_interval     = 24 *  3600.0  # sec



In [5]:
# Master Config
config_master = dict(
    total_simulation_time = total_simulation_time,
    time_step = master_time_step,
)

# Atmosphere model
config_atm = ComponentConfig(
    name = "atm",
    timestep = master_time_step,
    substeps = atm_substeps,
    save_interval = atm_save_interval,
    grid = None,
    params = None,
)

model_atm = Speedy(
    config = config_atm,
)

# Ocean model
config_ocn = ComponentConfig(
    name = "ocn",
    timestep = master_time_step,
    substeps = ocn_substeps,
    save_interval = ocn_save_interval,
    grid = None,
    params = None,
)
model_ocn = SlabOceanModel(config_ocn)

# Flux model
config_flx = ComponentConfig(
    name = "flx",
    timestep = master_time_step,
    substeps = flx_substeps,
    save_interval = flx_save_interval,
    grid = None,
    params = None,
)
model_flx = FluxModel(config_flx)

In [ ]:
master = Master(
    config = config_master,
    components = dict(
        flx = model_flx,
        atm = model_atm,
        ocn = model_ocn,
    ), 
)

## Register Variable Alias Names

In [ ]:
master.data_center.registerAlias(
    varname_universal = "surface_air_temperature",
    component         = "atm",
    varname_component = "surface_temperature",
)

master.data_center.registerAlias(
    varname_universal = "sea_surface_temperature",
    component         = "ocn",
    varname_component = "T",
)

In [ ]:
master.checkPlan()
master.printPlan()

import time

t0 = time.time()

state = model_atm.model.get_initial_state()
#final_state, predictions = model.unroll(state)

t1 = time.time()

total_time = t1-t0
print(f"Total time: {total_time:.1f} s.")

Print execution plan:
[ 1] : flx 
[ 2] : atm 
[ 3] : ocn 
Total time: 0.0 s.


In [7]:
record_atm_varnames = [
    "normalized_surface_pressure",
    "u_wind",
    "v_wind", 
    "specific_humidity", 
    "temperature", 
    "shortwave_rad.cloudc",
    "shortwave_rad.qcloud",
    "shortwave_rad.icltop",
    "shortwave_rad.cloudstr",
]
atm_recorder = { varname : [] for varname in record_atm_varnames }

def record_atm(step):
    atm = master.components["atm"]
    pred_ds = atm.model.predictions_to_xarray(atm.speedy_holder["tmp_pred"])
    for varname in record_atm_varnames:
        ds = pred_ds[varname]
        t = ds.coords["time"].to_numpy()
        t += pd.Timedelta(days=step)
        ds = ds.assign_coords(dict(
            time = t,
        ))
        atm_recorder[varname].append(ds)



# Run the Model

In [ ]:
for step in range(master_steps):
    print(f"Master Step: {step:d}/{master_steps:d}")
    master.run()
    record_atm(step)



Master Step: 0/60
Master Step: 1/60
Master Step: 2/60
Master Step: 3/60
Master Step: 4/60
Master Step: 5/60
Master Step: 6/60
Master Step: 7/60


In [ ]:
print("Merge")
merged = dict()
for varname in record_atm_varnames:
    merged[varname] = xr.merge(atm_recorder[varname])[varname]


In [ ]:
merged["normalized_surface_pressure"]

# Plot the Results

In [ ]:
#import matplotlib.pyplot as plt

merged['normalized_surface_pressure'].plot.contourf(x='lon', y='lat', col='time', col_wrap=3, aspect=2)

In [ ]:
merged['u_wind'].mean('lon').plot(x='lat', y='level', col='time', col_wrap=3, aspect=6, yincrease=False)
merged['u_wind'].isel(level=-1).plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)

In [ ]:
merged['v_wind'].mean('lon').plot(x='lat', y='level', col='time', col_wrap=3, aspect=6, yincrease=False)
merged['v_wind'].isel(level=-1).plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)

In [ ]:
merged['temperature'].mean('lon').plot(x='lat', y='level', col='time', col_wrap=2, aspect=6, yincrease=False)

In [ ]:
merged['specific_humidity'].mean('lon').plot(x='lat', y='level', col='time', col_wrap=3, aspect=6, yincrease=False)
merged['specific_humidity'].isel(level=3).plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)

In [ ]:
merged['shortwave_rad.cloudc'].plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)
merged['shortwave_rad.qcloud'].plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)
merged['shortwave_rad.icltop'].plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)
merged['shortwave_rad.cloudstr'].plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)

###### 